<a href="https://colab.research.google.com/github/Kishoby/Churn_Prediction_ML/blob/Model-Training/Churn_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import os
import time
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

In [2]:
split_path = (
    "/content/drive/MyDrive/Project_Churn Prediction/"
    "03_Train_Test_Split"
)

model_path = (
    "/content/drive/MyDrive/Project_Churn Prediction/"
    "04_Model_Training"
)

os.makedirs(model_path, exist_ok=True)

print("Model training output folder:")
print(model_path)

Model training output folder:
/content/drive/MyDrive/Project_Churn Prediction/04_Model_Training


In [3]:
X_train = pd.read_csv(
    f"{split_path}/X_train.csv"
)

y_train = pd.read_csv(
    f"{split_path}/y_train.csv"
).squeeze("columns")

print("Training data loaded successfully.")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

Training data loaded successfully.
X_train shape: (5625, 58)
y_train shape: (5625,)


In [4]:
display(X_train.head())

print("\nTarget distribution:")
display(
    y_train.value_counts()
    .sort_index()
    .to_frame("Count")
)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingServiceCount,TotalAdditionalServices,NoSecurityOrTechSupport,HasInternetService,FiberOpticCustomer,HighRiskBillingProfile,ChargesPerContractMonth,Tenure_MonthlyCharges_Interaction,Tenure_ServiceCount_Interaction,MonthlyCharges_ServiceCount_Interaction
0,0,65,94.55,6078.75,True,True,True,True,False,True,...,0,5,0,1,1,0,3.939583,6145.75,325,472.75
1,0,26,35.75,1022.50,True,False,False,False,True,False,...,0,2,0,1,0,0,35.750000,929.50,52,71.50
2,0,68,90.20,6297.65,False,True,False,True,False,True,...,0,4,0,1,1,0,3.758333,6133.60,272,360.80
3,0,3,84.30,235.05,True,False,False,True,False,False,...,1,2,1,1,1,1,84.300000,252.90,6,168.60
4,0,49,40.65,2070.75,False,True,False,False,True,False,...,1,2,0,1,0,0,40.650000,1991.85,98,81.30



Target distribution:


,Count
Churn,
0,4130
1,1495


In [5]:
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in y_train:", y_train.isnull().sum())

print(
    "Infinite values in X_train:",
    np.isinf(
        X_train.select_dtypes(include=np.number)
    ).sum().sum()
)

print("Target classes:", sorted(y_train.unique()))

Missing values in X_train: 0
Missing values in y_train: 0
Infinite values in X_train: 0
Target classes: [np.int64(0), np.int64(1)]


In [6]:
if X_train.isnull().sum().sum() > 0:
    raise ValueError("X_train contains missing values.")

if y_train.isnull().sum() > 0:
    raise ValueError("y_train contains missing values.")

if np.isinf(
    X_train.select_dtypes(include=np.number)
).sum().sum() > 0:
    raise ValueError("X_train contains infinite values.")

if set(y_train.unique()) != {0, 1}:
    raise ValueError(
        "The Churn target must contain only 0 and 1."
    )

print("Training data validation completed successfully.")

Training data validation completed successfully.


In [7]:
models = {
    "Logistic Regression": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    )
}

print("Models defined successfully:")

for model_name in models:
    print("-", model_name)

Models defined successfully:
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting


In [8]:
training_results = []

trained_models = {}

model_filenames = {
    "Logistic Regression":
        "Logistic_Regression_Model.pkl",

    "Decision Tree":
        "Decision_Tree_Model.pkl",

    "Random Forest":
        "Random_Forest_Model.pkl",

    "Gradient Boosting":
        "Gradient_Boosting_Model.pkl"
}

for model_name, model in models.items():

    print("=" * 60)
    print(f"Training: {model_name}")
    print("=" * 60)

    start_time = time.time()

    model.fit(
        X_train,
        y_train
    )

    end_time = time.time()

    training_time = end_time - start_time

    trained_models[model_name] = model

    model_file = os.path.join(
        model_path,
        model_filenames[model_name]
    )

    joblib.dump(
        model,
        model_file
    )

    training_results.append({
        "Model": model_name,
        "Training_Rows": X_train.shape[0],
        "Number_of_Features": X_train.shape[1],
        "Training_Time_Seconds": round(
            training_time,
            4
        ),
        "Model_File": model_filenames[model_name],
        "Training_Status": "Completed"
    })

    print(f"{model_name} trained successfully.")
    print(
        "Training time:",
        round(training_time, 4),
        "seconds"
    )
    print("Saved as:", model_file)

Training: Logistic Regression
Logistic Regression trained successfully.
Training time: 0.5464 seconds
Saved as: /content/drive/MyDrive/Project_Churn Prediction/04_Model_Training/Logistic_Regression_Model.pkl
Training: Decision Tree
Decision Tree trained successfully.
Training time: 0.1003 seconds
Saved as: /content/drive/MyDrive/Project_Churn Prediction/04_Model_Training/Decision_Tree_Model.pkl
Training: Random Forest
Random Forest trained successfully.
Training time: 3.8014 seconds
Saved as: /content/drive/MyDrive/Project_Churn Prediction/04_Model_Training/Random_Forest_Model.pkl
Training: Gradient Boosting
Gradient Boosting trained successfully.
Training time: 7.0774 seconds
Saved as: /content/drive/MyDrive/Project_Churn Prediction/04_Model_Training/Gradient_Boosting_Model.pkl


In [9]:
training_summary = pd.DataFrame(
    training_results
)

display(training_summary)

,Model,Training_Rows,Number_of_Features,Training_Time_Seconds,Model_File,Training_Status
0,Logistic Regression,5625,58,0.5464,Logistic_Regression_Model.pkl,Completed
1,Decision Tree,5625,58,0.1003,Decision_Tree_Model.pkl,Completed
2,Random Forest,5625,58,3.8014,Random_Forest_Model.pkl,Completed
3,Gradient Boosting,5625,58,7.0774,Gradient_Boosting_Model.pkl,Completed


In [10]:
training_summary.to_csv(
    f"{model_path}/Model_Training_Summary.csv",
    index=False
)

print("Model training summary saved successfully.")

Model training summary saved successfully.


In [11]:
feature_names = pd.DataFrame({
    "Feature_Order": range(
        1,
        len(X_train.columns) + 1
    ),
    "Feature_Name": X_train.columns
})

feature_names.to_csv(
    f"{model_path}/Model_Feature_Names.csv",
    index=False
)

display(feature_names.head())

,Feature_Order,Feature_Name
0,1,SeniorCitizen
1,2,tenure
2,3,MonthlyCharges
3,4,TotalCharges
4,5,gender_Male


In [12]:
model_configuration = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],

    "Main Configuration": [
        (
            "StandardScaler, max_iter=2000, "
            "class_weight=balanced"
        ),
        (
            "max_depth=8, min_samples_split=10, "
            "min_samples_leaf=5, class_weight=balanced"
        ),
        (
            "n_estimators=300, min_samples_split=5, "
            "min_samples_leaf=2, class_weight=balanced"
        ),
        (
            "n_estimators=200, learning_rate=0.05, "
            "max_depth=3"
        )
    ],

    "Purpose": [
        "Linear baseline classification model",
        "Interpretable tree-based classification model",
        "Ensemble model using multiple decision trees",
        "Sequential boosting classification model"
    ]
})

model_configuration.to_csv(
    f"{model_path}/Model_Configuration.csv",
    index=False
)

display(model_configuration)

,Model,Main Configuration,Purpose
0,Logistic Regression,"StandardScaler, max_iter=2000, class_weight=ba...",Linear baseline classification model
1,Decision Tree,"max_depth=8, min_samples_split=10, min_samples...",Interpretable tree-based classification model
2,Random Forest,"n_estimators=300, min_samples_split=5, min_sam...",Ensemble model using multiple decision trees
3,Gradient Boosting,"n_estimators=200, learning_rate=0.05, max_depth=3",Sequential boosting classification model


In [13]:
expected_files = [
    "Logistic_Regression_Model.pkl",
    "Decision_Tree_Model.pkl",
    "Random_Forest_Model.pkl",
    "Gradient_Boosting_Model.pkl",
    "Model_Training_Summary.csv",
    "Model_Feature_Names.csv",
    "Model_Configuration.csv"
]

print("Saved output verification:\n")

for filename in expected_files:

    full_path = os.path.join(
        model_path,
        filename
    )

    if os.path.exists(full_path):
        print(f"✓ {filename}")
    else:
        print(f"✗ Missing: {filename}")

Saved output verification:

✓ Logistic_Regression_Model.pkl
✓ Decision_Tree_Model.pkl
✓ Random_Forest_Model.pkl
✓ Gradient_Boosting_Model.pkl
✓ Model_Training_Summary.csv
✓ Model_Feature_Names.csv
✓ Model_Configuration.csv


In [14]:
for model_name, filename in model_filenames.items():

    loaded_model = joblib.load(
        os.path.join(
            model_path,
            filename
        )
    )

    print(
        f"{model_name} loaded successfully:",
        type(loaded_model).__name__
    )

Logistic Regression loaded successfully: Pipeline
Decision Tree loaded successfully: DecisionTreeClassifier
Random Forest loaded successfully: RandomForestClassifier
Gradient Boosting loaded successfully: GradientBoostingClassifier


In [15]:
import shutil

zip_file = "/content/Model_Training_Outputs"

shutil.make_archive(
    zip_file,
    "zip",
    model_path
)

print("ZIP file created successfully:")
print(f"{zip_file}.zip")

ZIP file created successfully:
/content/Model_Training_Outputs.zip


In [16]:
from google.colab import files

files.download(
    "/content/Model_Training_Outputs.zip"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>